# **Baseline CNN Model - Version 1**

# *Block 1: Libraries*


In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import cv2
from PIL import Image, ImageDraw
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import time
from datetime import datetime
from collections import Counter

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")

# Set random seeds
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

# Check device
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"# {num_gpus} GPU(s) available")
    for i in range(num_gpus):
        print(f"   GPU {i}: {torch.cuda.get_device_name(i)}")
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
    print(f"# Using CPU")

# 2 GPU(s) available
   GPU 0: Tesla T4
   GPU 1: Tesla T4


# *Block 2: Cobb Angle Calculation from Landmarks*

In [2]:
# ============================================
# COBB ANGLE CALCULATION FROM VERTEBRA CORNERS
# ============================================

def calculate_cobb_angle_from_corners(upper_corners, lower_corners):
    """
    Calculate Cobb angle from upper and lower endplate corners of a vertebra
    
    Args:
        upper_corners: [x1, y1, x2, y2] for upper endplate
        lower_corners: [x1, y1, x2, y2] for lower endplate
    
    Returns:
        angle: Angle in degrees
    """
    # Calculate slope of upper endplate
    upper_slope = (upper_corners[3] - upper_corners[1]) / (upper_corners[2] - upper_corners[0] + 1e-6)
    upper_angle = np.arctan(upper_slope) * 180 / np.pi
    
    # Calculate slope of lower endplate
    lower_slope = (lower_corners[3] - lower_corners[1]) / (lower_corners[2] - lower_corners[0] + 1e-6)
    lower_angle = np.arctan(lower_slope) * 180 / np.pi
    
    # Cobb angle is the absolute difference
    cobb_angle = abs(upper_angle - lower_angle)
    
    return cobb_angle


def get_end_vertebrae_tilt(vertebra_corners):
    """
    Get the tilt angle of each vertebra from its four corners
    
    Args:
        vertebra_corners: List of [x1,y1, x2,y2, x3,y3, x4,y4] for each vertebra
    
    Returns:
        tilt_angles: List of tilt angles for each vertebra
        centers: List of center points for each vertebra
    """
    tilt_angles = []
    centers = []
    
    for corners in vertebra_corners:
        if len(corners) >= 8:
            # Extract four corners
            x1, y1, x2, y2, x3, y3, x4, y4 = corners[:8]
            
            # Calculate center of vertebra
            center_x = (x1 + x2 + x3 + x4) / 4
            center_y = (y1 + y2 + y3 + y4) / 4
            centers.append((center_x, center_y))
            
            # Calculate mid-line slope (average of upper and lower endplates)
            upper_slope = (y2 - y1) / (x2 - x1 + 1e-6)
            lower_slope = (y3 - y4) / (x3 - x4 + 1e-6)
            avg_slope = (upper_slope + lower_slope) / 2
            
            tilt_angle = np.arctan(avg_slope) * 180 / np.pi
            tilt_angles.append(tilt_angle)
    
    return tilt_angles, centers


def calculate_full_cobb_angles(vertebra_corners, tilt_angles, centers):
    """
    Calculate PT, MT, and TL Cobb angles from vertebra tilts
    
    Returns:
        pt_angle, mt_angle, tl_angle: Cobb angles for each curve
        pt_end_vertebrae, mt_end_vertebrae, tl_end_vertebrae: Indices of end vertebrae
    """
    if len(tilt_angles) < 4:
        return 0, 0, 0, [], [], []
    
    # Smooth tilt angles to reduce noise
    smoothed = np.convolve(tilt_angles, np.ones(3)/3, mode='same')
    
    # Find local maxima and minima (curve inflection points)
    diffs = np.diff(smoothed)
    sign_changes = np.where(np.diff(np.sign(diffs)))[0]
    
    # Identify end vertebrae for each curve
    total_vertebrae = len(tilt_angles)
    
    # PT Curve (Proximal Thoracic - upper spine, ~T1-T6)
    pt_start = 0
    pt_end = min(6, total_vertebrae // 3)
    pt_angle = abs(max(tilt_angles[pt_start:pt_end+1]) - min(tilt_angles[pt_start:pt_end+1]))
    pt_end_vertebrae = [pt_start + np.argmax(tilt_angles[pt_start:pt_end+1]), 
                        pt_start + np.argmin(tilt_angles[pt_start:pt_end+1])]
    
    # MT Curve (Main Thoracic - mid spine, ~T6-T12)
    mt_start = max(0, total_vertebrae // 4)
    mt_end = min(2 * total_vertebrae // 3, total_vertebrae - 4)
    mt_angle = abs(max(tilt_angles[mt_start:mt_end+1]) - min(tilt_angles[mt_start:mt_end+1]))
    mt_end_vertebrae = [mt_start + np.argmax(tilt_angles[mt_start:mt_end+1]),
                        mt_start + np.argmin(tilt_angles[mt_start:mt_end+1])]
    
    # TL Curve (Thoracolumbar - lower spine, ~T12-L5)
    tl_start = max(0, 2 * total_vertebrae // 3)
    tl_end = total_vertebrae - 1
    tl_angle = abs(max(tilt_angles[tl_start:tl_end+1]) - min(tilt_angles[tl_start:tl_end+1]))
    tl_end_vertebrae = [tl_start + np.argmax(tilt_angles[tl_start:tl_end+1]),
                        tl_start + np.argmin(tilt_angles[tl_start:tl_end+1])]
    
    return pt_angle, mt_angle, tl_angle, pt_end_vertebrae, mt_end_vertebrae, tl_end_vertebrae

# *Block 3: Dataset Class for Vertebra Detection*

In [3]:
# ============================================
# DATASET CLASS FOR VERTEBRA DETECTION
# ============================================

class VertebraDetectionDataset(Dataset):
    """
    Dataset that learns to detect vertebra bounding boxes and corners
    This is the foundation for Cobb angle calculation
    """

    def __init__(self, image_dir, labels_file, target_size=(512, 512),
                 use_dynamic_padding=False, transform=None):
        
        self.image_dir = Path(image_dir)
        self.target_size = target_size
        self.use_dynamic_padding = use_dynamic_padding
        self.transform = transform

        # Load labels
        with open(labels_file, 'r') as f:
            self.json_data = json.load(f)

        print(f"# Loading dataset from: {image_dir}")

        # Prepare image-label pairs
        self.image_files = []
        self.bboxes_list = []
        self.corners_list = []
        self.cobb_angles_pt = []
        self.cobb_angles_mt = []
        self.cobb_angles_tl = []
        self.severities = []

        for img_name, label_data in self.json_data.items():
            img_path = self.image_dir / img_name

            if img_path.exists():
                self.image_files.append(img_path)

                # 1. Extract basic labels (Bboxes, segmentations)
                self.bboxes_list.append(label_data.get('bboxes', []))
                
                # Flatten segmentations for the corners tensor
                segmentations = label_data.get('segmentations', [])
                current_image_corners = []
                for seg in segmentations:
                    flat_seg = np.array(seg).flatten().tolist()
                    if len(flat_seg) >= 8:
                        current_image_corners.append(flat_seg[:8])
                self.corners_list.append(current_image_corners)

                # 2. THE UPGRADE: Grab the exact angles directly from JSON!
                # Default to [0,0,0] if missing
                angles = label_data.get('angles', [0.0, 0.0, 0.0]) 
                pt, mt, tl = angles[0], angles[1], angles[2]

                self.cobb_angles_pt.append(pt)
                self.cobb_angles_mt.append(mt)
                self.cobb_angles_tl.append(tl)
                
                # 3. Determine Overall Max Angle and Severity
                max_angle = max(pt, mt, tl)
                
                if max_angle < 10:
                    self.severities.append(0)  # Normal
                elif max_angle < 25:
                    self.severities.append(1)  # Mild
                elif max_angle < 45:
                    self.severities.append(2)  # Severe
                else:
                    self.severities.append(3)  # Very Severe

        # Overall list for the trainer
        self.cobb_angles = [
            max(p, m, t)
            for p, m, t in zip(
                self.cobb_angles_pt,
                self.cobb_angles_mt,
                self.cobb_angles_tl
            )
        ]

        print(f"# Loaded {len(self.image_files)} images")

        if len(self.cobb_angles) > 0:
            print(f"   Cobb angle range: {min(self.cobb_angles):.1f}° - {max(self.cobb_angles):.1f}°")
            
            class_counts = Counter(self.severities)
            severity_names = ["Normal", "Mild", "Severe", "Very Severe"]

            for cls in range(4):
                count = class_counts.get(cls, 0)

                print(f"   {severity_names[cls]}: {count} ({count/len(self.severities)*100:.1f}%)")

    def _dynamic_padding(self, image):
        """Apply dynamic padding to preserve aspect ratio"""

        h, w = image.shape[:2]

        target_h, target_w = self.target_size

        scale = min(target_h / h, target_w / w)

        new_h = int(h * scale)
        new_w = int(w * scale)

        resized = cv2.resize(
            image,
            (new_w, new_h),
            interpolation=cv2.INTER_CUBIC
        )

        canvas = np.zeros(
            (target_h, target_w, 3),
            dtype=np.uint8
        )

        y_offset = (target_h - new_h) // 2
        x_offset = (target_w - new_w) // 2

        canvas[
            y_offset:y_offset + new_h,
            x_offset:x_offset + new_w
        ] = resized

        return canvas

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):

        # Load image
        img_path = self.image_files[idx]

        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        original_h, original_w = image.shape[:2]

        # Apply padding/resizing
        if self.use_dynamic_padding:
            image = self._dynamic_padding(image)

        else:
            image = cv2.resize(image, self.target_size)

        # Normalize
        image = image.astype(np.float32) / 255.0

        # Apply transforms
        if self.transform:
            transformed = self.transform(image=image)
            image = transformed['image']

        else:
            image = torch.from_numpy(image).permute(2, 0, 1)

        # Labels
        cobb_angle = torch.tensor(
            self.cobb_angles[idx],
            dtype=torch.float32
        )

        severity = torch.tensor(
            self.severities[idx],
            dtype=torch.long
        )

        # Return bboxes and corners
        bboxes = (
            torch.tensor(
                self.bboxes_list[idx],
                dtype=torch.float32
            )
            if self.bboxes_list[idx]
            else torch.zeros(0, 4)
        )

        corners = (
            torch.tensor(
                self.corners_list[idx],
                dtype=torch.float32
            )
            if self.corners_list[idx]
            else torch.zeros(0, 8)
        )

        return {
            'image': image,
            'severity': severity,
            'cobb_angle': cobb_angle,
            'bboxes': bboxes,
            'corners': corners,
            'image_path': str(img_path),
            'original_size': (original_h, original_w)
        }

# *Block 4: CNN Architecture (From Scratch)*
**Model Architecture for Vertebra Detection**


In [4]:
# ============================================
# MODEL ARCHITECTURE FOR VERTEBRA DETECTION
# ============================================

class VertebraCNN(nn.Module):
    """
    CNN that outputs:
    1. Severity classification (4 classes)
    2. Cobb angle regression
    3. Vertebra bounding boxes (optional)
    4. Vertebra corner heatmaps (optional)
    """
    
    def __init__(self, num_classes=4, dropout_rate=0.3):
        super(VertebraCNN, self).__init__()
        
        # Encoder blocks
        self.enc1 = self._make_layer(3, 64, dropout_rate)
        self.enc2 = self._make_layer(64, 128, dropout_rate)
        self.enc3 = self._make_layer(128, 256, dropout_rate)
        self.enc4 = self._make_layer(256, 512, dropout_rate)
        self.enc5 = self._make_layer(512, 512, dropout_rate, pool=False)
        
        # Global Average Pooling
        self.gap = nn.AdaptiveAvgPool2d(1)
        
        # Shared FC layers
        self.fc1 = nn.Linear(512, 256)
        self.fc2 = nn.Linear(256, 128)
        self.dropout = nn.Dropout(dropout_rate)
        
        # Classification head
        self.classifier = nn.Linear(128, num_classes)
        
        # Regression head (Cobb angle)
        self.regressor = nn.Linear(128, 1)
        
        # Optional: Vertebra count head
        self.vertebra_counter = nn.Linear(128, 1)
        
        self._initialize_weights()
    
    def _make_layer(self, in_ch, out_ch, dropout, pool=True):
        layers = [
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        if pool:
            layers.append(nn.MaxPool2d(2))
        return nn.Sequential(*layers)
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        # Encoder
        x = self.enc1(x)
        x = self.enc2(x)
        x = self.enc3(x)
        x = self.enc4(x)
        x = self.enc5(x)
        
        # Global pooling
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        
        # Shared layers
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        
        # Outputs
        class_out = self.classifier(x)
        regress_out = self.regressor(x)
        vertebra_count = torch.sigmoid(self.vertebra_counter(x))
        
        return class_out, regress_out, vertebra_count
    
    def get_model_config(self):
        total_params = sum(p.numel() for p in self.parameters())
        return {
            'architecture': 'VertebraCNN',
            'total_parameters': total_params,
            'num_layers': len(list(self.modules())),
            'dropout_rate': 0.3,
            'input_shape': '3x512x512',
            'outputs': ['Classification (4)', 'Regression (1)', 'Vertebra Count']
        }

# *Block 5: Trainer Class with Comprehensive Metrics*

In [5]:
# ============================================
# TRAINER WITH 2-PHASE SUPPORT & METRICS FIX
# ============================================

import torch.nn.functional as F

class TrainerV1:
    def __init__(self, model, device, save_dir='./phase1_results_v2'):
        self.model = model
        self.device = device
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        
        self.history = {
            'train_loss': [], 'val_loss': [],
            'train_class_loss': [], 'val_class_loss': [],
            'train_reg_loss': [], 'val_reg_loss': [],
            'train_acc': [], 'val_acc': [],
            'train_mae': [], 'val_mae': [],
            'train_f1': [], 'val_f1': [],
            'lr': [], 'epoch_time': []
        }
        
        self.best_val_mae = float('inf')
        self.best_val_acc = 0.0
        self.best_epoch = 0
        
    def compute_class_weights(self, dataset):
        """Compute weights for class imbalance, handling PyTorch Subsets properly."""
        
        # FIXED: Check if the dataset is wrapped in a PyTorch Subset
        if hasattr(dataset, 'dataset') and hasattr(dataset, 'indices'):
            # It is a Subset. Look at the original dataset and only grab the indices for this split.
            severities = [dataset.dataset.severities[i] for i in dataset.indices]
        else:
            # It is the raw dataset. Grab everything normally.
            severities = [dataset.severities[i] for i in range(len(dataset))]
            
        class_counts = Counter(severities)
        total = len(severities)
        
        # Calculate weights safely (using .get() to avoid KeyError if a class is completely missing)
        weights = {cls: total / (len(class_counts) * count) for cls, count in class_counts.items()}
        weight_tensor = torch.tensor([weights.get(i, 0.0) for i in range(4)], dtype=torch.float32).to(self.device)
        
        print("\n# Class Distribution & Weights:")
        severity_names = ["Normal", "Mild", "Severe", "Very Severe"]
        for cls in range(4):
            count = class_counts.get(cls, 0)
            weight = weights.get(cls, 0.0)
            print(f"   {severity_names[cls]}: {count} ({count/total*100:.1f}%) - Weight: {weight:.3f}")
        
        return weight_tensor
    
    # ADDED: regress_weight parameter
    def train_epoch(self, train_loader, optimizer, criterion_class, criterion_regress, class_weights, regress_weight):
        self.model.train()
        
        epoch_loss = 0.0
        epoch_class_loss = 0.0
        epoch_reg_loss = 0.0
        all_preds, all_labels, all_cobb_preds, all_cobb_true = [], [], [], []
        
        for batch in tqdm(train_loader, desc="Training"):
            images = batch['image'].to(self.device)
            severities = batch['severity'].to(self.device)
            cobb_angles = batch['cobb_angle'].to(self.device)
            
            optimizer.zero_grad()
            class_out, regress_out, _ = self.model(images)
            
            if class_weights is not None:
                criterion_class.weight = class_weights
            
            class_loss = criterion_class(class_out, severities)
            regress_loss = criterion_regress(regress_out.squeeze(), cobb_angles)
            
            # APPLIED: Dynamic regression weighting
            total_loss = class_loss + (regress_weight * regress_loss)
            
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            optimizer.step()
            
            epoch_loss += total_loss.item()
            epoch_class_loss += class_loss.item()
            epoch_reg_loss += regress_loss.item()
            
            _, preds = torch.max(class_out, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(severities.cpu().numpy())
            all_cobb_preds.extend(regress_out.squeeze().detach().cpu().numpy())
            all_cobb_true.extend(cobb_angles.cpu().numpy())
        
        # FIXED: Cast numpy scalars to standard Python floats for JSON saving
        accuracy = float(accuracy_score(all_labels, all_preds))
        f1 = float(f1_score(all_labels, all_preds, average='weighted', zero_division=0))
        mae = float(np.mean(np.abs(np.array(all_cobb_true) - np.array(all_cobb_preds))))
        
        return {
            'loss': epoch_loss / len(train_loader),
            'class_loss': epoch_class_loss / len(train_loader),
            'reg_loss': epoch_reg_loss / len(train_loader),
            'accuracy': accuracy,
            'f1': f1,
            'mae': mae
        }
    
    # ADDED: regress_weight parameter
    def validate_epoch(self, val_loader, criterion_class, criterion_regress, regress_weight):
        self.model.eval()
        
        epoch_loss = 0.0
        all_preds, all_labels, all_cobb_preds, all_cobb_true = [], [], [], []
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validation"):
                images = batch['image'].to(self.device)
                severities = batch['severity'].to(self.device)
                cobb_angles = batch['cobb_angle'].to(self.device)
                
                class_out, regress_out, _ = self.model(images)
                
                class_loss = criterion_class(class_out, severities)
                regress_loss = criterion_regress(regress_out.squeeze(), cobb_angles)
                
                # APPLIED: Dynamic regression weighting
                total_loss = class_loss + (regress_weight * regress_loss)
                
                epoch_loss += total_loss.item()
                
                _, preds = torch.max(class_out, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(severities.cpu().numpy())
                all_cobb_preds.extend(regress_out.squeeze().cpu().numpy())
                all_cobb_true.extend(cobb_angles.cpu().numpy())
        
        # FIXED: Cast numpy scalars to standard Python floats
        accuracy = float(accuracy_score(all_labels, all_preds))
        f1 = float(f1_score(all_labels, all_preds, average='weighted', zero_division=0))
        mae = float(np.mean(np.abs(np.array(all_cobb_true) - np.array(all_cobb_preds))))
        
        return {
            'loss': epoch_loss / len(val_loader),
            'accuracy': accuracy,
            'f1': f1,
            'mae': mae,
            'predictions': all_preds,
            'true_labels': all_labels,
            'cobb_preds': all_cobb_preds,
            'cobb_true': all_cobb_true
        }
    
    # ADDED: regress_weight parameter with a default
    def train(self, train_loader, val_loader, epochs=5, lr=0.001, 
              weight_decay=1e-4, patience=15, class_weights=None, regress_weight=0.05):
        
        criterion_class = nn.CrossEntropyLoss()
        criterion_regress = nn.SmoothL1Loss(beta=1.0)
        optimizer = optim.AdamW(self.model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
        
        if torch.cuda.device_count() > 1 and not isinstance(self.model, nn.DataParallel):
            print(f"# Using DataParallel with {torch.cuda.device_count()} GPUs")
            self.model = nn.DataParallel(self.model)
        
        print(f"\n{'='*80}")
        print(f"TRAINING PHASE (Regress Weight: {regress_weight})")
        print(f"Epochs: {epochs}, LR: {lr}")
        print(f"{'='*80}\n")
        
        patience_counter = 0
        start_time = time.time()
        
        for epoch in range(epochs):
            epoch_start = time.time()
            print(f"\nEpoch {epoch+1}/{epochs}")
            print("-" * 50)
            
            # Pass regress_weight down
            train_metrics = self.train_epoch(train_loader, optimizer, criterion_class, 
                                             criterion_regress, class_weights, regress_weight)
            val_metrics = self.validate_epoch(val_loader, criterion_class, criterion_regress, regress_weight)
            
            scheduler.step(val_metrics['loss'])
            current_lr = optimizer.param_groups[0]['lr']
            
            epoch_time = time.time() - epoch_start
            
            # Save history safely
            for key in train_metrics:
                self.history.setdefault(f'train_{key}', []).append(train_metrics[key])
            for key in val_metrics:
                if key not in ['predictions', 'true_labels', 'cobb_preds', 'cobb_true']:
                    self.history.setdefault(f'val_{key}', []).append(val_metrics[key])
                    
            self.history.setdefault('lr', []).append(current_lr)
            self.history.setdefault('epoch_time', []).append(epoch_time)
            
            print(f"TRAIN | Loss: {train_metrics['loss']:.4f} | Acc: {train_metrics['accuracy']:.2%} | MAE: {train_metrics['mae']:.2f}° | F1: {train_metrics['f1']:.4f}")
            print(f"VAL   | Loss: {val_metrics['loss']:.4f} | Acc: {val_metrics['accuracy']:.2%} | MAE: {val_metrics['mae']:.2f}° | F1: {val_metrics['f1']:.4f}")
            print(f"Time: {epoch_time:.1f}s | LR: {current_lr:.6f}")
            
            # We track best model primarily by F1 score now, since classification is the hard part!
            if val_metrics['f1'] > self.best_val_acc or val_metrics['mae'] < self.best_val_mae:
                self.best_val_mae = min(self.best_val_mae, val_metrics['mae'])
                self.best_val_acc = max(self.best_val_acc, val_metrics['f1'])
                self.best_epoch = len(self.history['train_loss']) # absolute epoch number
                patience_counter = 0
                
                model_to_save = self.model.module if hasattr(self.model, 'module') else self.model
                torch.save({
                    'epoch': self.best_epoch,
                    'model_state_dict': model_to_save.state_dict(),
                    'val_mae': val_metrics['mae'],
                    'val_acc': val_metrics['accuracy'],
                    'val_f1': val_metrics['f1'],
                }, self.save_dir / 'best_model_v1.pth')
                print(f"  ✓ Saved best model!")
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"\n# Early stopping triggered.")
                    break
                    
        return self.history

# *Block 6: Visualization and Comparative Analysis*

In [6]:
# ============================================
# VISUALIZATION AND COMPARATIVE ANALYSIS
# ============================================

from sklearn.metrics import precision_recall_fscore_support, roc_curve, auc, r2_score, mean_squared_error
from sklearn.preprocessing import label_binarize

class ComparativeAnalyzer:
    def __init__(self, trainer, save_dir, model_name='BaselineCNN_V1'):
        self.trainer = trainer
        self.save_dir = Path(save_dir)
        self.model_name = model_name
        self.save_dir.mkdir(parents=True, exist_ok=True)
    
    def plot_learning_curves(self):
        """Plot training and validation curves"""
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        
        # Loss curves
        axes[0, 0].plot(self.trainer.history['train_loss'], label='Train', linewidth=2)
        axes[0, 0].plot(self.trainer.history['val_loss'], label='Validation', linewidth=2)
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].set_title('Total Loss')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # Accuracy
        axes[0, 1].plot(self.trainer.history['train_accuracy'], label='Train', linewidth=2)
        axes[0, 1].plot(self.trainer.history['val_accuracy'], label='Validation', linewidth=2)
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Accuracy')
        axes[0, 1].set_title('Classification Accuracy')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # MAE
        axes[0, 2].plot(self.trainer.history['train_mae'], label='Train', linewidth=2)
        axes[0, 2].plot(self.trainer.history['val_mae'], label='Validation', linewidth=2)
        axes[0, 2].axhline(y=5, color='r', linestyle='--', label='Clinical Goal (5°)')
        axes[0, 2].set_xlabel('Epoch')
        axes[0, 2].set_ylabel('MAE (degrees)')
        axes[0, 2].set_title('Cobb Angle Prediction Error')
        axes[0, 2].legend()
        axes[0, 2].grid(True, alpha=0.3)
        
        # F1 Score
        axes[1, 0].plot(self.trainer.history['train_f1'], label='Train', linewidth=2)
        axes[1, 0].plot(self.trainer.history['val_f1'], label='Validation', linewidth=2)
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('F1 Score')
        axes[1, 0].set_title('Weighted F1 Score')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # Learning Rate
        axes[1, 2].plot(self.trainer.history['lr'], linewidth=2, color='purple')
        axes[1, 2].set_xlabel('Epoch')
        axes[1, 2].set_ylabel('Learning Rate')
        axes[1, 2].set_title('Learning Rate Schedule')
        axes[1, 2].set_yscale('log')
        axes[1, 2].grid(True, alpha=0.3)
        
        # Remove empty subplot (1,1) since we merged precision/recall elsewhere
        axes[1, 1].axis('off')
        
        plt.tight_layout()
        plt.savefig(self.save_dir / f'{self.model_name}_learning_curves.png', dpi=150)
        plt.close()
        print(f"✓ Learning curves saved to {self.save_dir}")
    
    def plot_confusion_matrix(self, true_labels, predictions):
        """Plot confusion matrix"""
        cm = confusion_matrix(true_labels, predictions)
        severity_labels = ['Normal\n(<10°)', 'Mild\n(10-25°)', 'Severe\n(25-45°)', 'Very Severe\n(>45°)']
        
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=severity_labels, yticklabels=severity_labels)
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.title(f'{self.model_name} - Confusion Matrix')
        plt.tight_layout()
        plt.savefig(self.save_dir / f'{self.model_name}_confusion_matrix.png', dpi=150)
        plt.close()
        
        # Save confusion matrix as CSV
        cm_df = pd.DataFrame(cm, index=severity_labels, columns=severity_labels)
        cm_df.to_csv(self.save_dir / f'{self.model_name}_confusion_matrix.csv')
        print(f"✓ Confusion matrix saved")
    
    def plot_roc_curves(self, true_labels, predictions, num_classes=4):
        """Plot ROC curves for each class"""
        # Binarize labels
        y_true_bin = label_binarize(true_labels, classes=[0, 1, 2, 3])
        
        # Get prediction probabilities (using model probabilities)
        y_pred_bin = label_binarize(predictions, classes=[0, 1, 2, 3])
        
        plt.figure(figsize=(10, 8))
        severity_labels = ['Normal', 'Mild', 'Severe', 'Very Severe']
        
        for i in range(num_classes):
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_bin[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, linewidth=2, label=f'{severity_labels[i]} (AUC = {roc_auc:.2f})')
        
        plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'{self.model_name} - ROC Curves')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(self.save_dir / f'{self.model_name}_roc_curves.png', dpi=150)
        plt.close()
        print(f"✓ ROC curves saved")
    
    def plot_regression_analysis(self, true_cobb, pred_cobb):
        """Plot regression analysis for Cobb angle predictions"""
        fig, axes = plt.subplots(2, 2, figsize=(14, 12))
        
        # Scatter plot
        axes[0, 0].scatter(true_cobb, pred_cobb, alpha=0.5)
        axes[0, 0].plot([0, 60], [0, 60], 'r--', linewidth=2, label='Perfect Prediction')
        axes[0, 0].set_xlabel('True Cobb Angle (degrees)')
        axes[0, 0].set_ylabel('Predicted Cobb Angle (degrees)')
        axes[0, 0].set_title('Predicted vs True Cobb Angles')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # Residual plot
        residuals = np.array(true_cobb) - np.array(pred_cobb)
        axes[0, 1].scatter(pred_cobb, residuals, alpha=0.5)
        axes[0, 1].axhline(y=0, color='r', linestyle='--', linewidth=2)
        axes[0, 1].set_xlabel('Predicted Cobb Angle (degrees)')
        axes[0, 1].set_ylabel('Residual (degrees)')
        axes[0, 1].set_title('Residual Plot')
        axes[0, 1].grid(True, alpha=0.3)
        
        # Error distribution
        axes[1, 0].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
        axes[1, 0].set_xlabel('Prediction Error (degrees)')
        axes[1, 0].set_ylabel('Frequency')
        axes[1, 0].set_title('Error Distribution')
        axes[1, 0].axvline(x=0, color='r', linestyle='--')
        axes[1, 0].axvline(x=5, color='g', linestyle='--', label='Clinical Goal (±5°)')
        axes[1, 0].axvline(x=-5, color='g', linestyle='--')
        axes[1, 0].legend()
        
        # MAE by severity range
        ranges = [(0, 10), (10, 25), (25, 45), (45, 100)]
        range_labels = ['Normal', 'Mild', 'Severe', 'Very Severe']
        mae_by_range = []
        
        for (low, high) in ranges:
            mask = (np.array(true_cobb) >= low) & (np.array(true_cobb) < high)
            if np.sum(mask) > 0:
                mae = np.mean(np.abs(np.array(true_cobb)[mask] - np.array(pred_cobb)[mask]))
            else:
                mae = 0
            mae_by_range.append(mae)
        
        axes[1, 1].bar(range_labels, mae_by_range, color='skyblue')
        axes[1, 1].axhline(y=5, color='r', linestyle='--', linewidth=2, label='Clinical Goal')
        axes[1, 1].set_xlabel('Severity')
        axes[1, 1].set_ylabel('MAE (degrees)')
        axes[1, 1].set_title('MAE by Severity Range')
        axes[1, 1].legend()
        
        plt.tight_layout()
        plt.savefig(self.save_dir / f'{self.model_name}_regression_analysis.png', dpi=150)
        plt.close()
        print(f"✓ Regression analysis saved")
    
    def save_metrics_table(self, val_metrics):
        """Save comprehensive metrics table"""
        metrics_table = {
            'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'MAE (degrees)', 'RMSE (degrees)'],
            'Value': [
                f"{val_metrics['accuracy']:.4f}",
                f"{val_metrics.get('precision', 0):.4f}",
                f"{val_metrics.get('recall', 0):.4f}",
                f"{val_metrics.get('f1', 0):.4f}",
                f"{val_metrics.get('mae', 0):.2f}",
                f"{val_metrics.get('rmse', 0):.2f}"
            ]
        }
        
        df = pd.DataFrame(metrics_table)
        df.to_csv(self.save_dir / f'{self.model_name}_metrics_table.csv', index=False)
        
        # Save training history
        if self.trainer.history:
            max_len = max([len(v) for v in self.trainer.history.values() if isinstance(v, list)])
            clean_history = {k: v for k, v in self.trainer.history.items() 
                           if isinstance(v, list) and len(v) == max_len}
            history_df = pd.DataFrame(clean_history)
            history_df.to_csv(self.save_dir / f'{self.model_name}_training_history.csv', index=False)
        
        print(f"✓ Metrics table saved")
        return df
    
    def save_model_config(self, model):
        """Save model configuration"""
        config = model.get_model_config()
        config['training_time_seconds'] = sum(self.trainer.history.get('epoch_time', [0]))
        config['total_epochs'] = len(self.trainer.history.get('train_loss', []))
        config['best_epoch'] = getattr(self.trainer, 'best_epoch', 0)
        config['best_val_mae'] = getattr(self.trainer, 'best_val_mae', 0)
        config['best_val_acc'] = getattr(self.trainer, 'best_val_acc', 0)
        
        with open(self.save_dir / f'{self.model_name}_config.json', 'w') as f:
            json.dump(config, f, indent=2)
        
        print(f"✓ Model configuration saved")
        return config
    
    def generate_all_reports(self, val_loader, criterion_class, criterion_regress, regress_weight=0.05):
        """Generate all visualizations and reports"""
        print("\n# Generating all visualizations and reports...")
        # Get validation metrics
        val_metrics = self.trainer.validate_epoch(val_loader, criterion_class, criterion_regress, regress_weight)
        
        # Generate all plots
        self.plot_learning_curves()
        self.plot_confusion_matrix(val_metrics['true_labels'], val_metrics['predictions'])
        self.plot_roc_curves(val_metrics['true_labels'], val_metrics['predictions'])
        self.plot_regression_analysis(val_metrics['cobb_true'], val_metrics['cobb_preds'])
        
        # Save metrics and config
        self.save_metrics_table(val_metrics)
        self.save_model_config(self.trainer.model.module if hasattr(self.trainer.model, 'module') else self.trainer.model)
        
        print(f"\n✓ All reports saved to: {self.save_dir}")
        
        return val_metrics

# *Block 7: Main Execution with Dual GPU*

In [7]:
# ============================================
# MAIN EXECUTION 
# ============================================

def vertebra_collate_fn(batch):
    """Custom collate function to handle variable numbers of bounding boxes."""
    # 1. Stack items that are consistently sized
    images = torch.stack([item['image'] for item in batch])
    severities = torch.stack([item['severity'] for item in batch])
    cobb_angles = torch.stack([item['cobb_angle'] for item in batch])
    
    # 2. Keep variable-length items as lists
    bboxes = [item['bboxes'] for item in batch]
    corners = [item['corners'] for item in batch]
    
    # 3. Keep metadata as lists
    image_paths = [item['image_path'] for item in batch]
    original_sizes = [item['original_size'] for item in batch]
    
    return {
        'image': images,
        'severity': severities,
        'cobb_angle': cobb_angles,
        'bboxes': bboxes,
        'corners': corners,
        'image_path': image_paths,
        'original_size': original_sizes
    }
def main_v1():
    print("="*80)
    print("VERSION 1: VERTEBRA DETECTION & COBB ANGLE CALCULATION")
    print("="*80)
    
    # Paths for Phase 1 dataset
    PHASE1_IMAGE_DIR = "/kaggle/input/datasets/ruturajkasture/final-batch1-dataset5/Images1/Images1"
    PHASE1_LABELS_FILE = "/kaggle/input/datasets/ruturajkasture/final-batch1-dataset5/Labels1/Labels1/Balanced_Labels.json"

    # Check if paths exist
    if not os.path.exists(PHASE1_IMAGE_DIR):
        print(f"# Image directory not found: {PHASE1_IMAGE_DIR}")
        return
    if not os.path.exists(PHASE1_LABELS_FILE):
        print(f"# Labels file not found: {PHASE1_LABELS_FILE}")
        return
    
    # Create dataset
    print("\n# Loading Phase 1 Dataset...")
    dataset = VertebraDetectionDataset(
        image_dir=PHASE1_IMAGE_DIR,
        labels_file=PHASE1_LABELS_FILE,
        target_size=(512, 512),
        use_dynamic_padding=False,
        transform=None
    )
    
    # Split dataset
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
    
    # Create data loaders
    batch_size = 16
    if torch.cuda.device_count() > 1:
        batch_size = batch_size * torch.cuda.device_count()
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                             num_workers=4, pin_memory=True, collate_fn=vertebra_collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, 
                           num_workers=4, pin_memory=True, collate_fn=vertebra_collate_fn)
    
    print(f"\n# Dataset Split:")
    print(f"   Train: {train_size} images")
    print(f"   Validation: {val_size} images")
    print(f"   Batch Size: {batch_size}")
    
    # Initialize model
    model = VertebraCNN(num_classes=4, dropout_rate=0.3)
    model = model.to(device)
    
    # Print model summary
    config = model.get_model_config()
    print(f"\n# Model Configuration:")
    print(f"   Architecture: {config['architecture']}")
    print(f"   Parameters: {config['total_parameters']:,}")
    
    # Trainer
    # Initialize trainer
    trainer = TrainerV1(model, device, save_dir='./phase1_results_v2')
    class_weights = trainer.compute_class_weights(train_dataset)

    # ========================================
    # PHASE 1: CLASSIFICATION FOCUS
    # ========================================
    print("\n" + "="*80)
    print("PHASE 1: TRAINING CLASSIFIER (REGRESSION MUTED)")
    print("="*80)
    # regress_weight = 0.0 means the model will ONLY care about classification accuracy
    history_p1 = trainer.train(
        train_loader, val_loader, 
        epochs=30, 
        lr=0.001, 
        class_weights=class_weights, 
        regress_weight=0.0 
    )

    # ========================================
    # PHASE 2: JOINT FINE-TUNING
    # ========================================
    print("\n" + "="*80)
    print("PHASE 2: FINE-TUNING CLASSIFIER & REGRESSOR")
    print("="*80)
    # regress_weight = 0.05 turns the angle calculation back on without overpowering classification
    # Notice we drop the learning rate (lr=0.0001) for fine-tuning
    history_p2 = trainer.train(
        train_loader, val_loader, 
        epochs=15, 
        lr=0.0001, 
        class_weights=class_weights, 
        regress_weight=0.05 
    )

    # ========================================
    # GENERATING REPORTS
    # ========================================
    print("\n" + "=" * 80)
    print("# GENERATING COMPLETE ANALYSIS REPORTS")
    print("=" * 80)
    
    # We need to recreate the criterion objects just for the final report generation
    criterion_class = nn.CrossEntropyLoss(weight=class_weights)
    criterion_regress = nn.SmoothL1Loss(beta=1.0)
    
    analyzer = ComparativeAnalyzer(trainer=trainer, save_dir='./phase1_results_v2/analysis', model_name='BaselineCNN_V1')
    
    # The analyzer validate_epoch call needs the regress_weight passed to it now!
    val_metrics = trainer.validate_epoch(val_loader, criterion_class, criterion_regress, regress_weight=0.05)
    
    # ========================================
    # GENERATE ALL VISUALIZATIONS AND REPORTS
    # ========================================
    
    print("\n" + "="*80)
    print("# GENERATING COMPLETE ANALYSIS REPORTS")
    print("="*80)
    
    # Create analyzer
    analyzer = ComparativeAnalyzer(
        trainer=trainer, 
        save_dir='./phase1_results_v2/analysis', 
        model_name='BaselineCNN_V1'
    )
    
    # Generate all reports
    final_val_metrics = analyzer.generate_all_reports(val_loader, criterion_class, criterion_regress, regress_weight=0.05)
    # ========================================
    # PRINT FINAL SUMMARY WITH FILE LOCATIONS
    # ========================================
    
    print("\n" + "="*80)
    print("# ALL FILES SAVED SUCCESSFULLY!")
    print("="*80)
    
    files_saved = {
        'Model Weights': [
            './phase1_results_v2/best_model_v1.pth',
            './phase1_results_v2/final_model_v1.pth'
        ],
        'Training Data': [
            './phase1_results_v2/analysis/BaselineCNN_V1_training_history.csv',
            './phase1_results_v2/analysis/BaselineCNN_V1_metrics_table.csv',
            './phase1_results_v2/analysis/BaselineCNN_V1_config.json'
        ],
        'Visualizations': [
            './phase1_results_v2/analysis/BaselineCNN_V1_learning_curves.png',
            './phase1_results_v2/analysis/BaselineCNN_V1_confusion_matrix.png',
            './phase1_results_v2/analysis/BaselineCNN_V1_roc_curves.png',
            './phase1_results_v2/analysis/BaselineCNN_V1_regression_analysis.png'
        ]
    }
    
    for category, files in files_saved.items():
        print(f"\n# {category}:")
        for f in files:
            if os.path.exists(f):
                print(f"   # {f}")
            else:
                print(f"   # {f} (not found)")
    
    # Print final metrics
    print("\n" + "="*80)
    print("# FINAL PERFORMANCE METRICS")
    print("="*80)
    print(f"# Best Validation MAE: {trainer.best_val_mae:.2f}°")
    print(f"# Best Validation Accuracy: {trainer.best_val_acc:.2%}")
    print(f"# Best Validation F1 Score: {max(trainer.history['val_f1']):.4f}")
    print(f"# Final Validation Accuracy: {final_val_metrics['accuracy']:.2%}")
    print(f"# Final Validation MAE: {final_val_metrics['mae']:.2f}°")
    
    print("\n" + "="*80)
    print("# VERSION 1 COMPLETE! READY FOR VERSION 2 (FINE-TUNING)")
    print("="*80)
    
    return trainer, model, trainer.history

if __name__ == "__main__":
    trainer_v1, model_v1, history_v1 = main_v1()

VERSION 1: VERTEBRA DETECTION & COBB ANGLE CALCULATION

# Loading Phase 1 Dataset...
# Loading dataset from: /kaggle/input/datasets/ruturajkasture/final-batch1-dataset5/Images1/Images1
# Loaded 1991 images
   Cobb angle range: 3.7° - 56.0°
   Normal: 491 (24.7%)
   Mild: 500 (25.1%)
   Severe: 500 (25.1%)
   Very Severe: 500 (25.1%)

# Dataset Split:
   Train: 1592 images
   Validation: 399 images
   Batch Size: 32

# Model Configuration:
   Architecture: VertebraCNN
   Parameters: 9,575,878

# Class Distribution & Weights:
   Normal: 384 (24.1%) - Weight: 1.036
   Mild: 406 (25.5%) - Weight: 0.980
   Severe: 395 (24.8%) - Weight: 1.008
   Very Severe: 407 (25.6%) - Weight: 0.978

PHASE 1: TRAINING CLASSIFIER (REGRESSION MUTED)
# Using DataParallel with 2 GPUs

TRAINING PHASE (Regress Weight: 0.0)
Epochs: 30, LR: 0.001


Epoch 1/30
--------------------------------------------------


Validation: 100%|██████████| 13/13 [00:06<00:00,  2.14it/s]


TRAIN | Loss: 1.4395 | Acc: 25.75% | MAE: 23.98° | F1: 0.2550
VAL   | Loss: 1.4122 | Acc: 25.31% | MAE: 23.14° | F1: 0.1269
Time: 70.8s | LR: 0.001000
  ✓ Saved best model!

Epoch 2/30
--------------------------------------------------


Validation: 100%|██████████| 13/13 [00:05<00:00,  2.22it/s]


TRAIN | Loss: 1.4039 | Acc: 28.96% | MAE: 23.93° | F1: 0.2849
VAL   | Loss: 1.3127 | Acc: 34.59% | MAE: 23.07° | F1: 0.2654
Time: 70.3s | LR: 0.001000
  ✓ Saved best model!

Epoch 3/30
--------------------------------------------------


...

TRAIN | Loss: 1.0406 | Acc: 64.82% | MAE: 7.38° | F1: 0.6423
VAL   | Loss: 0.7231 | Acc: 73.43% | MAE: 5.55° | F1: 0.7188
Time: 73.0s | LR: 0.000100
  ✓ Saved best model!

# GENERATING COMPLETE ANALYSIS REPORTS


Validation: 100%|██████████| 13/13 [00:05<00:00,  2.25it/s]



# GENERATING COMPLETE ANALYSIS REPORTS

# Generating all visualizations and reports...


Validation: 100%|██████████| 13/13 [00:05<00:00,  2.17it/s]


✓ Learning curves saved to phase1_results_v2/analysis
✓ Confusion matrix saved
✓ ROC curves saved
✓ Regression analysis saved
✓ Metrics table saved
✓ Model configuration saved

✓ All reports saved to: phase1_results_v2/analysis

# ALL FILES SAVED SUCCESSFULLY!

# Model Weights:
   # ./phase1_results_v2/best_model_v1.pth
   # ./phase1_results_v2/final_model_v1.pth (not found)

# Training Data:
   # ./phase1_results_v2/analysis/BaselineCNN_V1_training_history.csv
   # ./phase1_results_v2/analysis/BaselineCNN_V1_metrics_table.csv
   # ./phase1_results_v2/analysis/BaselineCNN_V1_config.json

# Visualizations:
   # ./phase1_results_v2/analysis/BaselineCNN_V1_learning_curves.png
   # ./phase1_results_v2/analysis/BaselineCNN_V1_confusion_matrix.png
   # ./phase1_results_v2/analysis/BaselineCNN_V1_roc_curves.png
   # ./phase1_results_v2/analysis/BaselineCNN_V1_regression_analysis.png

# FINAL PERFORMANCE METRICS
# Best Validation MAE: 5.55°
# Best Validation Accuracy: 75.18%
# Best Validation 

# *Testing cell*